# AI Studio LLM Evaluation Results Analysis
Analyzing `result.json` from the prompts evaluation

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from pathlib import Path

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 1. Load result.json

In [ ]:
result_file = Path('prompts/result.json')

with open(result_file, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

print(f'Loaded {len(raw_data)} trace evaluations')

In [ ]:
# Convert to DataFrame
records = []

for item in raw_data:
    trace_id = item.get('trace_id', '')
    answer_found = item.get('answer_found', {})
    step_index = answer_found.get('step_index', -1)
    worker = answer_found.get('worker', 'Unknown')
    evidence = answer_found.get('evidence', '')
    final_judgement = item.get('final_judgement', '')
    tools_involved = answer_found.get('tools_involved', [])
    tool_names = [t.get('tool', '') for t in tools_involved]
    tool_steps = [t.get('step', 0) for t in tools_involved]
    num_tools = len(tools_involved)
    found = step_index >= 0
    
    records.append({
        'trace_id': trace_id,
        'step_index': step_index,
        'worker': worker,
        'found': found,
        'num_tools': num_tools,
        'tools': tool_names,
        'tool_steps': tool_steps,
        'evidence': evidence,
        'final_judgement': final_judgement,
        'evidence_length': len(evidence),
        'judgement_length': len(final_judgement)
    })

df = pd.DataFrame(records)
df_found = df[df['found'] == True].copy()
df_failed = df[df['found'] == False].copy()

print(f'Successful: {len(df_found)} | Failed: {len(df_failed)}')
df.head()

## 2. Overall Success Rate

In [ ]:
total = len(df)
found_count = df['found'].sum()
not_found_count = total - found_count

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Pie chart
colors = ['#2ecc71', '#e74c3c']
axes[0].pie([found_count, not_found_count], 
            labels=['Found', 'Not Found'],
            autopct='%1.1f%%',
            colors=colors,
            explode=(0.05, 0.05),
            shadow=True,
            startangle=90)
axes[0].set_title('Answer Found Rate', fontsize=14, fontweight='bold')

# Bar chart
axes[1].bar(['Found', 'Not Found'], [found_count, not_found_count], color=colors)
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution of Results', fontsize=14, fontweight='bold')
for i, v in enumerate([found_count, not_found_count]):
    axes[1].text(i, v + 1, str(v), ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('result_analysis_success_rate.png', dpi=150)
plt.show()

## 3. Step Index Distribution

In [ ]:
# Statistics
print('Step Index Statistics (successful traces):')
print(df_found['step_index'].describe())

sorted_steps = sorted(df_found['step_index'])
q1 = sorted_steps[int(len(sorted_steps)*0.25)]
q2 = sorted_steps[int(len(sorted_steps)*0.50)]
q3 = sorted_steps[int(len(sorted_steps)*0.75)]

print(f'\nQ1: {q1} | Median: {q2} | Q3: {q3}')
print(f'IQR: {q3-q1}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogram
axes[0, 0].hist(df_found['step_index'], bins=25, color='#3498db', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df_found['step_index'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df_found["step_index"].mean():.1f}')
axes[0, 0].axvline(df_found['step_index'].median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {df_found["step_index"].median():.0f}')
axes[0, 0].set_xlabel('Step Index')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Step Indices', fontweight='bold')
axes[0, 0].legend()

# Box plot
axes[0, 1].boxplot(df_found['step_index'], vert=True, patch_artist=True)
axes[0, 1].set_ylabel('Step Index')
axes[0, 1].set_title('Step Index Box Plot', fontweight='bold')

# Performance buckets
buckets = {
    'Very Fast (≤15)': len([s for s in df_found['step_index'] if s <= 15]),
    'Fast (16-25)': len([s for s in df_found['step_index'] if 16 <= s <= 25]),
    'Normal (26-35)': len([s for s in df_found['step_index'] if 26 <= s <= 35]),
    'Slow (36-45)': len([s for s in df_found['step_index'] if 36 <= s <= 45]),
    'Very Slow (>45)': len([s for s in df_found['step_index'] if s > 45]),
}

bucket_colors = ['#27ae60', '#3498db', '#f39c12', '#e74c3c', '#c0392b']
axes[1, 0].bar(buckets.keys(), buckets.values(), color=bucket_colors, edgecolor='black')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Performance Bucket Distribution', fontweight='bold')
axes[1, 0].tick_params(axis='x', rotation=15)

# Cumulative distribution
sorted_steps = np.sort(df_found['step_index'])
cumulative = np.arange(1, len(sorted_steps) + 1) / len(sorted_steps)
axes[1, 1].plot(sorted_steps, cumulative, color='#e67e22', linewidth=2)
axes[1, 1].fill_between(sorted_steps, cumulative, alpha=0.3, color='#e67e22')
axes[1, 1].set_xlabel('Step Index')
axes[1, 1].set_ylabel('Cumulative Proportion')
axes[1, 1].set_title('Cumulative Distribution', fontweight='bold')
for p in [25, 50, 75]:
    val = np.percentile(df_found['step_index'], p)
    axes[1, 1].axvline(val, color='red', linestyle=':', alpha=0.7)

plt.tight_layout()
plt.savefig('result_analysis_step_distribution.png', dpi=150)
plt.show()

## 4. Worker Performance Analysis

In [ ]:
worker_stats = defaultdict(lambda: {'found': 0, 'not_found': 0, 'steps': []})

for d in raw_data:
    worker = d['answer_found'].get('worker', 'Unknown')
    step = d['answer_found'].get('step_index', -1)
    if step >= 0:
        worker_stats[worker]['found'] += 1
        worker_stats[worker]['steps'].append(step)
    else:
        worker_stats[worker]['not_found'] += 1

worker_summary = []
for worker, stats in worker_stats.items():
    total_w = stats['found'] + stats['not_found']
    avg_step = sum(stats['steps'])/len(stats['steps']) if stats['steps'] else 0
    success_rate = stats['found']/total_w*100 if total_w > 0 else 0
    worker_summary.append({
        'worker': worker,
        'total': total_w,
        'found': stats['found'],
        'not_found': stats['not_found'],
        'success_rate': success_rate,
        'avg_step': avg_step,
    })

worker_df = pd.DataFrame(worker_summary).sort_values('total', ascending=False)
worker_df

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Top 10 workers by count
top_workers = worker_df.head(10)
axes[0].barh(top_workers['worker'][::-1], top_workers['total'][::-1], color='#8e44ad', edgecolor='black')
axes[0].set_xlabel('Number of Traces')
axes[0].set_title('Top 10 Workers by Frequency', fontweight='bold')

# Success rate
axes[1].barh(top_workers['worker'][::-1], top_workers['success_rate'][::-1], color='#16a085', edgecolor='black')
axes[1].set_xlabel('Success Rate (%)')
axes[1].set_title('Top 10 Workers - Success Rate', fontweight='bold')
axes[1].axvline(87.1, color='red', linestyle='--', linewidth=2, label='Overall (87.1%)')
axes[1].legend()

plt.tight_layout()
plt.savefig('result_analysis_worker_performance.png', dpi=150)
plt.show()

## 5. Tool Usage Analysis

In [ ]:
# Extract all tools
all_tools = []
for tools_list in df['tools']: 
    all_tools.extend(tools_list)

tool_counts = Counter(all_tools)

print(f'Total tool invocations: {len(all_tools)}')
print(f'Unique tools: {len(tool_counts)}')
print('\nTop 15 Most Used Tools:')
for tool, count in tool_counts.most_common(15):
    print(f'  {tool}: {count}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 15 tools bar chart
top_tools = dict(tool_counts.most_common(15))
axes[0].barh(list(top_tools.keys())[::-1], list(top_tools.values())[::-1], 
             color='#2980b9', edgecolor='black')
axes[0].set_xlabel('Usage Count')
axes[0].set_title('Top 15 Most Used Tools', fontweight='bold')

# Tool effectiveness
tool_success_rate = defaultdict(lambda: {'success': 0, 'total': 0})

for _, row in df.iterrows():
    for tool in row['tools']:
        tool_success_rate[tool]['total'] += 1
        if row['found']:
            tool_success_rate[tool]['success'] += 1

tool_effectiveness = pd.DataFrame([
    {'tool': tool, 'success': data['success'], 'total': data['total'], 
     'rate': data['success'] / data['total'] * 100 if data['total'] > 0 else 0}
    for tool, data in tool_success_rate.items()
]).sort_values('total', ascending=False)

top_eff = tool_effectiveness.head(12)
x = range(len(top_eff))
width = 0.35

bars1 = axes[1].bar([i - width/2 for i in x], top_eff['success'], width, label='Success', color='#27ae60')
bars2 = axes[1].bar([i + width/2 for i in x], top_eff['total'] - top_eff['success'], width, label='Failed', color='#e74c3c')

axes[1].set_ylabel('Count')
axes[1].set_title('Tool Success vs Failure Distribution', fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(top_eff['tool'], rotation=45, ha='right')
axes[1].legend()

plt.tight_layout()
plt.savefig('result_analysis_tool_usage.png', dpi=150)
plt.show()

## 6. Tool Efficiency Correlation

In [ ]:
# Tools used in fast vs slow traces
early_tools = Counter()
late_tools = Counter()

for _, row in df_found.iterrows():
    step = row['step_index']
    tools = row['tools']
    if step <= 25:
        for t in tools:
            early_tools[t] += 1
    else:
        for t in tools:
            late_tools[t] += 1

all_tools_set = set(early_tools.keys()) | set(late_tools.keys())
comparison_data = []
for tool in all_tools_set:
    comparison_data.append({
        'tool': tool,
        'fast': early_tools.get(tool, 0),
        'slow': late_tools.get(tool, 0)
    })

comparison_df = pd.DataFrame(comparison_data)
comparison_df['total'] = comparison_df['fast'] + comparison_df['slow']
comparison_df = comparison_df.sort_values('total', ascending=False).head(12)

fig, ax = plt.subplots(figsize=(12, 7))

x = range(len(comparison_df))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], comparison_df['fast'], width, label='Fast (≤25 steps)', color='#27ae60')
bars2 = ax.bar([i + width/2 for i in x], comparison_df['slow'], width, label='Slow (>25 steps)', color='#e74c3c')

ax.set_ylabel('Usage Count')
ax.set_title('Tool Usage: Fast vs Slow Traces', fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(comparison_df['tool'], rotation=45, ha='right')
ax.legend()

plt.tight_layout()
plt.savefig('result_analysis_tool_efficiency.png', dpi=150)
plt.show()

## 7. Failure Analysis

In [ ]:
print(f'Failed Traces: {len(df_failed)}')
print('\nFailed Trace IDs:')
for tid in df_failed['trace_id'].values:
    print(f'  {tid}')

In [ ]:
# Analyze failure evidence
failure_keywords = ['no relevant', 'not found', 'missing', 'failed', 'no content', 
                     'not present', 'no match', 'unavailable', 'could not', 'weather',
                     'engineering', 'material', 'fatigue', 'pressure', 'structural']

keyword_counts = Counter()
for evidence in df_failed['evidence']:
    evidence_lower = evidence.lower()
    for keyword in failure_keywords:
        if keyword in evidence_lower:
            keyword_counts[keyword] += 1

fig, ax = plt.subplots(figsize=(10, 6))

if keyword_counts:
    ax.barh(list(keyword_counts.keys())[::-1], list(keyword_counts.values())[::-1],
            color='#c0392b', edgecolor='black')
    ax.set_xlabel('Frequency')
    ax.set_title('Failure Keywords in Evidence', fontweight='bold')

plt.tight_layout()
plt.savefig('result_analysis_failure_keywords.png', dpi=150)
plt.show()

In [ ]:
# Worker failure distribution
failed_worker_counts = df_failed['worker'].value_counts()

fig, ax = plt.subplots(figsize=(8, 5))

ax.barh(failed_worker_counts.index[::-1], failed_worker_counts.values[::-1],
        color='#e74c3c', edgecolor='black')
ax.set_xlabel('Failed Count')
ax.set_title('Workers with Failed Traces', fontweight='bold')

plt.tight_layout()
plt.savefig('result_analysis_failed_workers.png', dpi=150)
plt.show()

## 8. Efficiency Insights

In [ ]:
# Single-tool vs Multi-tool traces
single_tool_count = len([d for d in df_found.iterrows() if len(d[1]['tools']) == 1])
multi_tool_count = len([d for d in df_found.iterrows() if len(d[1]['tools']) >= 3])

avg_step_single = df_found[df_found['num_tools'] == 1]['step_index'].mean()
avg_step_multi = df_found[df_found['num_tools'] >= 3]['step_index'].mean()

print('Single-Tool vs Multi-Tool Traces:')
print(f'  Single-tool traces: {single_tool_count} ({single_tool_count/len(df_found)*100:.1f}%)')
print(f'  Multi-tool (≥3) traces: {multi_tool_count} ({multi_tool_count/len(df_found)*100:.1f}%)')
print(f'  Avg step (single): {avg_step_single:.1f}')
print(f'  Avg step (multi): {avg_step_multi:.1f}')

In [ ]:
# Most efficient vs least efficient traces
efficient = df_found.nsmallest(5, 'step_index')
inefficient = df_found.nlargest(5, 'step_index')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Most efficient
axes[0].barh(efficient['trace_id'][::-1], efficient['step_index'][::-1], color='#27ae60', edgecolor='black')
axes[0].set_xlabel('Step Index')
axes[0].set_title('Most Efficient Traces (Earliest Found)', fontweight='bold')

# Least efficient
axes[1].barh(inefficient['trace_id'][::-1], inefficient['step_index'][::-1], color='#c0392b', edgecolor='black')
axes[1].set_xlabel('Step Index')
axes[1].set_title('Least Efficient Traces (Latest Found)', fontweight='bold')

plt.tight_layout()
plt.savefig('result_analysis_efficiency_comparison.png', dpi=150)
plt.show()

## 9. Correlation Heatmap

In [ ]:
correlation_data = df_found[['step_index', 'num_tools', 'evidence_length', 'judgement_length']].copy()
corr_matrix = correlation_data.corr()

fig, ax = plt.subplots(figsize=(8, 6))

sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=2, fmt='.2f', ax=ax)
ax.set_title('Correlation Heatmap', fontweight='bold')

plt.tight_layout()
plt.savefig('result_analysis_correlation.png', dpi=150)
plt.show()

## 10. Summary Statistics

In [ ]:
summary_stats = {
    'Metric': [
        'Total Traces',
        'Answers Found',
        'Answers Not Found',
        'Found Rate (%)',
        'Avg Step Index',
        'Min Step Index',
        'Max Step Index',
        'Median Step Index',
        'Std Step Index',
        'Avg Tools per Trace',
        'Most Common Tool',
        'Most Common Worker',
    ],
    'Value': [
        len(df),
        found_count,
        not_found_count,
        f'{found_count/len(df)*100:.1f}',
        f'{df_found["step_index"].mean():.1f}',
        df_found['step_index'].min(),
        df_found['step_index'].max(),
        df_found['step_index'].median(),
        f'{df_found["step_index"].std():.1f}',
        f'{df_found["num_tools"].mean():.1f}',
        tool_counts.most_common(1)[0][0] if tool_counts else 'N/A',
        worker_df.iloc[0]['worker'] if len(worker_df) > 0 else 'N/A',
    ]
}

summary_df = pd.DataFrame(summary_stats)
print('\nSUMMARY STATISTICS:')
print('=' * 50)
print(summary_df.to_string(index=False))

In [ ]:
# Export analysis results
output_dir = Path('analysis_output')
output_dir.mkdir(exist_ok=True)

df.to_csv(output_dir / 'processed_results.csv', index=False)
df_found.to_csv(output_dir / 'found_results.csv', index=False)
df_failed.to_csv(output_dir / 'failed_results.csv', index=False)
worker_df.to_csv(output_dir / 'worker_statistics.csv', index=False)
summary_df.to_csv(output_dir / 'summary_statistics.csv', index=False)

print(f'Results exported to: {output_dir}')

In [ ]:
print('\n' + '=' * 80)
print('ANALYSIS COMPLETE')
print('=' * 80)
print(f'\nKey Findings:')
print(f'1. {found_count/len(df)*100:.1f}% of traces successfully found answers')
print(f'2. Average step to find answer: {df_found["step_index"].mean():.1f}')
print(f'3. Most effective tool: {tool_counts.most_common(1)[0][0]} ({tool_counts.most_common(1)[0][1]} uses)')
print(f'4. Most active worker: {worker_df.iloc[0]["worker"]} ({worker_df.iloc[0]["total"]} traces)')
print(f'5. {len(df_failed)} traces failed - primarily due to missing content in corpus')
print(f'\nVisualizations saved as PNG files')
print(f'Data exports saved to analysis_output/')